In [44]:
import diff_diff
import pandas as pd
from diff_diff import CallawaySantAnna

In [45]:
print(diff_diff.get_llm_guide("practitioner"))

# diff-diff Practitioner Guide

> An 8-step workflow for rigorous Difference-in-Differences analysis, based on
> Baker et al. (2025) "Difference-in-Differences Designs: A Practitioner's
> Guide" and adapted for the diff-diff library. Some steps are reorganized or
> extended relative to the paper:
>
> - **Numbering**: diff-diff uses 1-Define, 2-Assumptions, 3-Test PT,
>   4-Choose estimator, 5-Estimate, 6-Sensitivity, 7-Heterogeneity,
>   8-Robustness. The paper uses 1-Define, 2-Assumptions, 3-Estimation method,
>   4-Uncertainty, 5-Estimate, 6-Sensitivity, 7-Heterogeneity, 8-Keep learning.
> - **Parallel trends testing** is a separate Step 3 (the paper embeds it in
>   Step 2), to ensure AI agents execute it as a distinct action.
> - **Sources of uncertainty** (paper's Step 4) are folded into Step 5
>   (Estimate) with an explicit cluster-count check directive: >= 50 clusters
>   for asymptotic SEs, otherwise wild bootstrap. The 50-cluster threshold is
>   a diff-diff convention.
> - *

In [46]:
panel = pd.read_csv("../data/processed/panel.csv")
panel["date"]=pd.to_datetime(panel["date"])
panel["first_launch"]=pd.to_datetime(panel["first_launch"])
print(panel['State'].nunique())

46


## Target Parameters
**Estimand**: Average Treatment Effect on the Treated (ATT) - Average effect of sports betting legalization on gambling helpline contacts per 100k, among states that legalized.

**Weighting**: Unweighted (equal weight per state) because helpline volume is normalized by population.

**Heterogeneity**: Do states that legalized early show different effects than states that legalized late?

**Event Study**: Dynamic effects by month relative to launch

## Identification Assumptions

**Parallel Trends**:  Each treated cohort follows parallel trends with never-treated states conditional on pre-PASPA baseline helpline volume. Implemented via the doubly robust estimator (consistent if either the outcome model or propensity score is correctly specified).

**No Anticipation**: Legalization doesn't affect call volume before the market opens because problem gambling behavior requires actual betting activity.

In [59]:
cs = CallawaySantAnna(
    control_group='never_treated',
    estimation_method='dr',
    cluster='State',
    n_bootstrap=999,
)

In [60]:
results_es = cs.fit(
    panel,
    outcome='contacts_per_100k',
    unit='State',
    time='period',
    first_treat='first_treat_period',
    covariates=['baseline_contacts_per_100k'],
    aggregate='event_study'
)

/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Low Events Per Variable (EPV = 1.0) in propensity score model for cohort g=30. 1 minority-class observations for 1 predictor variable(s). Peduzzi et al. (1996) recommend EPV >= 10. Estimates may be unreliable (overfitting, biased coefficients, inflated standard errors). Consider estimation_method='reg' to avoid propensity scores.
  beta_logistic, pscore = solve_logit(
/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Low Events Per Variable (EPV = 1.0) in propensity score model for cohort g=32. 1 minority-class observations for 1 predictor variable(s). Peduzzi et al. (1996) recommend EPV >= 10. Estimates may be unreliable (overfitting, biased coefficients, inflated standard errors). Consider estimation_method='reg' to avoid propensity scores.
  beta_logistic, pscore = solve_logit(


In [61]:
if results_es.event_study_effects:
    for rel_t, eff in sorted(results_es.event_study_effects.items()):
        if rel_t < 0:
            print(f"Pre-period {rel_t}: ATT={eff['effect']:.4f}, SE={eff['se']:.4f}")

Pre-period -97: ATT=1.1281, SE=0.9406
Pre-period -96: ATT=0.6212, SE=0.2719
Pre-period -95: ATT=-1.9110, SE=1.6950
Pre-period -94: ATT=2.5808, SE=2.0779
Pre-period -93: ATT=-1.7208, SE=1.5178
Pre-period -92: ATT=0.1747, SE=0.2358
Pre-period -91: ATT=-0.8534, SE=1.2442
Pre-period -90: ATT=-0.4782, SE=0.5380
Pre-period -89: ATT=-1.5848, SE=1.0352
Pre-period -88: ATT=2.0746, SE=0.8428
Pre-period -87: ATT=-1.5604, SE=0.7934
Pre-period -86: ATT=-0.1658, SE=0.3344
Pre-period -85: ATT=0.6241, SE=0.8800
Pre-period -84: ATT=0.2390, SE=0.5566
Pre-period -83: ATT=-0.2140, SE=0.3712
Pre-period -82: ATT=-0.5908, SE=0.5551
Pre-period -81: ATT=0.2066, SE=0.1972
Pre-period -80: ATT=0.0675, SE=0.4310
Pre-period -79: ATT=-0.0088, SE=0.2458
Pre-period -78: ATT=-0.2401, SE=0.2418
Pre-period -77: ATT=0.9758, SE=0.3388
Pre-period -76: ATT=0.4394, SE=0.5851
Pre-period -75: ATT=-0.9338, SE=0.6007
Pre-period -74: ATT=-0.0498, SE=0.3290
Pre-period -73: ATT=0.2122, SE=0.4455
Pre-period -72: ATT=0.8721, SE=0.8477

In [62]:
for rel_t, eff in sorted(results_es.event_study_effects.items()):
    if rel_t >= 0:
        print(f"Post-period {rel_t:4d}: ATT={eff['effect']:.4f}, SE={eff['se']:.4f}")

Post-period    0: ATT=1.8613, SE=0.6417
Post-period    1: ATT=2.7744, SE=1.0555
Post-period    2: ATT=1.3050, SE=0.5407
Post-period    3: ATT=1.0400, SE=0.5264
Post-period    4: ATT=1.0763, SE=0.5337
Post-period    5: ATT=0.4985, SE=0.3815
Post-period    6: ATT=0.8271, SE=0.5366
Post-period    7: ATT=0.1684, SE=0.4341
Post-period    8: ATT=0.5208, SE=0.5453
Post-period    9: ATT=0.6380, SE=0.5800
Post-period   10: ATT=0.2394, SE=0.6506
Post-period   11: ATT=0.7210, SE=0.6845
Post-period   12: ATT=0.1876, SE=0.6698
Post-period   13: ATT=-0.1508, SE=0.5736
Post-period   14: ATT=0.0476, SE=0.5419
Post-period   15: ATT=-0.1633, SE=0.5208
Post-period   16: ATT=0.2095, SE=0.5993
Post-period   17: ATT=0.4175, SE=0.7007
Post-period   18: ATT=0.8588, SE=0.7121
Post-period   19: ATT=0.8278, SE=0.6323
Post-period   20: ATT=0.8403, SE=0.6510
Post-period   21: ATT=0.7748, SE=0.7385
Post-period   22: ATT=0.6551, SE=0.7617
Post-period   23: ATT=0.7080, SE=0.6587
Post-period   24: ATT=0.8119, SE=0.633

In [63]:
print(f"Overall ATT: {results_es.overall_att:.4f}")
print(f"Overall SE:  {results_es.overall_se:.4f}")
print(results_es.summary())

Overall ATT: 1.3723
Overall SE:  0.5241
            Callaway-Sant'Anna Staggered Difference-in-Differences Results           

Total observations:                  5520
Treated units:                         27
Never-treated units:                   19
Treatment cohorts:                     21
Time periods:                         120
Control group:                 never_treated
Base period:                      varying

-------------------------------------------------------------------------------------
                   Overall Average Treatment Effect on the Treated                   
-------------------------------------------------------------------------------------
Parameter           Estimate    Std. Err.     t-stat      P>|t|   Sig.
-------------------------------------------------------------------------------------
ATT                   1.3723       0.5241      2.618     0.0080     **
-------------------------------------------------------------------------------------

95

In [64]:
results = cs.fit(
    panel,
    outcome='contacts_per_100k',
    unit='State',
    time='period',
    first_treat='first_treat_period',
    covariates=['baseline_contacts_per_100k'],
    aggregate='all',
)
print(results.summary())

/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Low Events Per Variable (EPV = 1.0) in propensity score model for cohort g=30. 1 minority-class observations for 1 predictor variable(s). Peduzzi et al. (1996) recommend EPV >= 10. Estimates may be unreliable (overfitting, biased coefficients, inflated standard errors). Consider estimation_method='reg' to avoid propensity scores.
  beta_logistic, pscore = solve_logit(
/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Low Events Per Variable (EPV = 1.0) in propensity score model for cohort g=32. 1 minority-class observations for 1 predictor variable(s). Peduzzi et al. (1996) recommend EPV >= 10. Estimates may be unreliable (overfitting, biased coefficients, inflated standard errors). Consider estimation_method='reg' to avoid propensity scores.
  beta_logistic, pscore = solve_logit(


            Callaway-Sant'Anna Staggered Difference-in-Differences Results           

Total observations:                  5520
Treated units:                         27
Never-treated units:                   19
Treatment cohorts:                     21
Time periods:                         120
Control group:                 never_treated
Base period:                      varying

-------------------------------------------------------------------------------------
                   Overall Average Treatment Effect on the Treated                   
-------------------------------------------------------------------------------------
Parameter           Estimate    Std. Err.     t-stat      P>|t|   Sig.
-------------------------------------------------------------------------------------
ATT                   1.3723       0.5312      2.583     0.0100      *
-------------------------------------------------------------------------------------

95% Confidence Interval: [0.3809, 2.4045]


/Users/kayvans/Documents/sports-betting-legalization/venv/lib/python3.14/site-packages/diff_diff/staggered.py:3419: UserWarning: Low Events Per Variable (EPV = 1.0) in propensity score model for cohort g=99. 1 minority-class observations for 1 predictor variable(s). Peduzzi et al. (1996) recommend EPV >= 10. Estimates may be unreliable (overfitting, biased coefficients, inflated standard errors). Consider estimation_method='reg' to avoid propensity scores.
  beta_logistic, pscore = solve_logit(
/var/folders/6p/5s7l8jtj05s_1p_1jxzg33480000gn/T/ipykernel_4954/1245163128.py:1: UserWarning: Low Events Per Variable (EPV) detected in propensity score estimation for 2499 of 2499 cell(s). Minimum EPV = 1.0 (cohort g=30). Consider estimation_method='reg' (avoids propensity scores) or reducing the number of covariates. See results.epv_summary() for details.
  results = cs.fit(
